## Text Classification with Tensorflow

This project uses neural networks to classify the sentiment of airline tweets (negative, neutral, positive). We build three text models in TensorFlow/Keras and compare them with the classic TF-IDF + Logistic Regression model from the earlier NLP project. The objective is in order to find out if a neural network can classify tweet sentiment better than a classic machine learning model, and understand why or why not.

## Approach
1. Load the data and clean the text (same steps as the earlier project)
2. Split the data with the same random seed, so the test set is identical and the results are comparable
3. Turn text into numbers: a word vocabulary (10,000 words) and sequences of 40 tokens
4. Build and train three networks: Embedding + Average Pooling, Conv1D, and a Bidirectional LSTM (with class weights and early stopping)
5. Compare the networks with the TF-IDF + Logistic Regression baseline using accuracy and macro F1
6. Look at the errors and summarize the findings
7. Show it in Streamlit

In [2]:
# Load the data
import pandas as pd
url = "https://raw.githubusercontent.com/ruchitgandhi/Twitter-Airline-Sentiment-Analysis/master/Tweets.csv"
df = pd.read_csv(url)[["text", "airline_sentiment"]]

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   text               14640 non-null  object
 1   airline_sentiment  14640 non-null  object
dtypes: object(2)
memory usage: 228.9+ KB


In [4]:
# tekrar edenler
df.text.duplicated().sum()

np.int64(213)

In [5]:
df.airline_sentiment.duplicated().sum()

np.int64(14637)

In [6]:
df = df.drop_duplicates(subset="text").reset_index(drop=True)

In [7]:
df.shape

(14427, 2)

In [9]:
df.head(1)

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral


In [10]:
# Clean the data
df["clean"] = (df["text"].str.lower()                                              # küçük harf
               .str.replace(r"http\S+|www\.\S+", " ", regex=True)                  # linkleri sil
               .str.replace(r"@\w+", " ", regex=True)                              # @mention'ları sil
               .str.replace(r"#", " ", regex=True)                                 # hashtag işaretini sil
               .str.replace(r"[^a-z\s']", " ", regex=True)                         # rakam, noktalama, emoji sil
               .str.replace(r"\s+", " ", regex=True).str.strip())                  # fazla boşlukları temizle
df.head()

,text,airline_sentiment,clean
0,@VirginAmerica What @dhepburn said.,neutral,what said
1,@VirginAmerica plus you've added commercials t...,positive,plus you've added commercials to the experienc...
2,@VirginAmerica I didn't today... Must mean I n...,neutral,i didn't today must mean i need to take anothe...
3,@VirginAmerica it's really aggressive to blast...,negative,it's really aggressive to blast obnoxious ente...
4,@VirginAmerica and it's a really big bad thing...,negative,and it's a really big bad thing about it


In [11]:
# Remove the empty fields
df = df[df["clean"].str.len() > 0].reset_index(drop=True)
print(df.shape)

(14427, 3)


In [12]:
# Splitting train and test
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    df["clean"], df["airline_sentiment"], test_size=0.20, random_state=42, stratify=df["airline_sentiment"])
print(x_train.shape, x_test.shape)

(11541,) (2886,)


In [13]:
# Creating catalog for sentiment
ytr, yte = (y.map({"negative": 0, "neutral": 1, "positive": 2}).values for y in (y_train, y_test))

In [16]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = dict(enumerate(compute_class_weight("balanced", classes=np.arange(3), y=ytr)))
print(class_weights)

{0: np.float64(0.5295980176211453), 1: np.float64(1.5734151329243353), 2: np.float64(2.0998908296943233)}


In [17]:
# Converting text to numeric
import tensorflow as tf
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)

vec = layers.TextVectorization(max_tokens=10000, output_sequence_length=40)
vec.adapt(x_train.values)

Xtr = vec(x_train.values).numpy()
Xte = vec(x_test.values).numpy()
print(Xtr.shape, Xte.shape)
print(vec.get_vocabulary()[:10])

(11541, 40) (2886, 40)
['', '[UNK]', np.str_('to'), np.str_('the'), np.str_('i'), np.str_('a'), np.str_('you'), np.str_('for'), np.str_('flight'), np.str_('on')]


In [18]:
# Train the model
def train(model):
    tf.keras.utils.set_random_seed(42)
    model.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
    model.fit(Xtr, ytr, epochs=12, batch_size=64, validation_split=0.1, class_weight=class_weights, callbacks=[stop], verbose=2)
    return model

In [19]:
# Embedding + Avarage Pooling
avg_model = train(tf.keras.Sequential([
    layers.Input((40,)),
    layers.Embedding(10000, 32),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(3),
]))

Epoch 1/12
163/163 - 3s - 18ms/step - accuracy: 0.4213 - loss: 1.0552 - val_accuracy: 0.6623 - val_loss: 0.9354
Epoch 2/12
163/163 - 2s - 10ms/step - accuracy: 0.6530 - loss: 0.9162 - val_accuracy: 0.7377 - val_loss: 0.7493
Epoch 3/12
163/163 - 2s - 11ms/step - accuracy: 0.7328 - loss: 0.7728 - val_accuracy: 0.7688 - val_loss: 0.6568
Epoch 4/12
163/163 - 1s - 5ms/step - accuracy: 0.7734 - loss: 0.6550 - val_accuracy: 0.7766 - val_loss: 0.5958
Epoch 5/12
163/163 - 1s - 6ms/step - accuracy: 0.8036 - loss: 0.5654 - val_accuracy: 0.7394 - val_loss: 0.6556
Epoch 6/12
163/163 - 1s - 8ms/step - accuracy: 0.8251 - loss: 0.5035 - val_accuracy: 0.7680 - val_loss: 0.5747
Epoch 7/12
163/163 - 1s - 5ms/step - accuracy: 0.8435 - loss: 0.4493 - val_accuracy: 0.7498 - val_loss: 0.6004
Epoch 8/12
163/163 - 1s - 6ms/step - accuracy: 0.8514 - loss: 0.4209 - val_accuracy: 0.7801 - val_loss: 0.5502
Epoch 9/12
163/163 - 1s - 5ms/step - accuracy: 0.8612 - loss: 0.3933 - val_accuracy: 0.7758 - val_loss: 0.557

In [20]:
# Conv1D
cnn_model = train(tf.keras.Sequential([
    layers.Input((40,)),
    layers.Embedding(10000, 32),
    layers.Conv1D(64, 3, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dropout(0.4),
    layers.Dense(3),
]))

Epoch 1/12
163/163 - 4s - 22ms/step - accuracy: 0.5846 - loss: 1.0135 - val_accuracy: 0.7359 - val_loss: 0.7726
Epoch 2/12
163/163 - 1s - 9ms/step - accuracy: 0.7311 - loss: 0.7362 - val_accuracy: 0.7593 - val_loss: 0.6106
Epoch 3/12
163/163 - 3s - 15ms/step - accuracy: 0.7919 - loss: 0.5720 - val_accuracy: 0.7740 - val_loss: 0.5709
Epoch 4/12
163/163 - 1s - 9ms/step - accuracy: 0.8401 - loss: 0.4501 - val_accuracy: 0.7835 - val_loss: 0.5584
Epoch 5/12
163/163 - 1s - 9ms/step - accuracy: 0.8735 - loss: 0.3571 - val_accuracy: 0.7835 - val_loss: 0.5792
Epoch 6/12
163/163 - 4s - 22ms/step - accuracy: 0.8981 - loss: 0.2856 - val_accuracy: 0.7758 - val_loss: 0.6163


In [21]:
# Bidirectional LSTM
lstm_model = train(tf.keras.Sequential([
    layers.Input((40,)),
    layers.Embedding(10000, 32, mask_zero=True),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dropout(0.4),
    layers.Dense(3),
]))

Epoch 1/12
163/163 - 13s - 78ms/step - accuracy: 0.6651 - loss: 0.9070 - val_accuracy: 0.7463 - val_loss: 0.6163
Epoch 2/12
163/163 - 7s - 42ms/step - accuracy: 0.8050 - loss: 0.5365 - val_accuracy: 0.7688 - val_loss: 0.5855
Epoch 3/12
163/163 - 11s - 67ms/step - accuracy: 0.8654 - loss: 0.3803 - val_accuracy: 0.7801 - val_loss: 0.6152
Epoch 4/12
163/163 - 8s - 50ms/step - accuracy: 0.9003 - loss: 0.2875 - val_accuracy: 0.7810 - val_loss: 0.6788


In [22]:
# Comparision
from sklearn.metrics import accuracy_score, f1_score

nets = {"Embedding + AvgPool": avg_model, "Conv1D": cnn_model, "BiLSTM": lstm_model}
preds = {n: m.predict(Xte, verbose=0).argmax(axis=1) for n, m in nets.items()}

table = pd.DataFrame({n: {"accuracy": accuracy_score(yte, p), "macro_f1": f1_score(yte, p, average="macro")} for n, p in preds.items()}).T
table.loc["TF-IDF + Logistic Regression (project 10)"] = [0.7952, 0.7489]
table.round(4).sort_values("macro_f1", ascending=False)

,accuracy,macro_f1
Embedding + AvgPool,0.7890,0.7494
TF-IDF + Logistic Regression (project 10),0.7952,0.7489
Conv1D,0.7852,0.7315
BiLSTM,0.7803,0.7286


In [24]:
from sklearn.metrics import classification_report
classes = ["negative", "neutral", "positive"]

print(classification_report(yte, preds["Embedding + AvgPool"], target_names=classes, digits=3))

              precision    recall  f1-score   support

    negative      0.906     0.816     0.859      1816
     neutral      0.579     0.734     0.647       612
    positive      0.727     0.758     0.742       458

    accuracy                          0.789      2886
   macro avg      0.738     0.769     0.749      2886
weighted avg      0.809     0.789     0.795      2886



In [25]:
# Save the model
inp = tf.keras.Input(shape=(1,), dtype=tf.string)
sentiment_model = tf.keras.Model(inp, avg_model(vec(inp)))
sentiment_model.save("sentiment_avgpool.keras")